**TÍTULO**

F3N2 — Classificação dos períodos de El Niño na região Niño 3.4

**CONTEXTO**

A definição operacional de El Niño (ONI/NOAA) usa médias trimestrais mensais de SSTA em Niño 3.4 com limiar de +0,5 °C por cinco estações sobrepostas. Esta fase trabalha em resolução semanal; o critério é adaptado preservando as escalas temporais originais, conforme a discussão de definição de Trenberth (1997).

**DESAFIO**

Hipótese: eventos de El Niño identificados na série semanal reproduzem o catálogo histórico (1982/83, 1997/98, 2015/16, 2023/24 entre os fortes) com datação mais fina de início, pico e término. Objetivos: (i) detectar eventos semanais; (ii) medir duração e intensidade; (iii) traçar os perfis temporais da SSTA média.

**METODOLOGIA**

SSTA de Niño 3.4 suavizada por média móvel centrada de 13 semanas (~3 meses, análogo semanal da média trimestral do ONI). Evento: SSTA suavizada ≥ +0,5 °C por pelo menos 22 semanas consecutivas (~5 meses, equivalente às 5 estações sobrepostas). Intensidade no pico define a classe: fraco (0,5–1,0), moderado (1,0–1,5), forte (1,5–2,0), muito forte (≥ 2,0 °C). Perfis alinhados pela semana do pico (lag 0) em janela de ±52 semanas.

**RESULTADOS ESPERADOS**

TabF3N2_catalogo_eventos.csv (início, fim, duração, pico, classe), FigF3N2_serie_eventos (série com eventos sombreados) e FigF3N2_perfis_eventos (evolução alinhada pelo pico), com tabelas numéricas associadas.

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. TRENBERTH, K. E. The Definition of El Niño. Bulletin of the American Meteorological Society, v. 78, p. 2771-2777, 1997. DOI: https://doi.org/10.1175/1520-0442(1997)010<2759:TDOENO>2.0.CO;2
2. HUANG, B. et al. Improvements of the Daily Optimum Interpolation Sea Surface Temperature (DOISST) Version 2.1. Journal of Climate, v. 34, p. 2923-2939, 2021. DOI: https://doi.org/10.1175/JCLI-D-20-0166.1

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

In [ ]:
ssta = pd.to_numeric(master['nino34_ssta'], errors='coerce')
suave = f3.smooth_ssta(ssta, 13)
catalogo = f3.detect_el_nino_events(ssta, smooth_weeks=13, threshold_c=0.5, min_duration_weeks=22)
f3.save_table(catalogo, 'TabF3N2_catalogo_eventos', index=False)
catalogo

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(ssta.index, ssta, color='0.6', lw=0.5, label='SSTA semanal')
ax.plot(suave.index, suave, color='crimson', lw=1.2, label='media movel 13 semanas')
ax.axhline(0.5, color='k', ls='--', lw=0.8, label='limiar +0,5 C')
for _, ev in catalogo.iterrows():
    ax.axvspan(pd.Timestamp(ev['inicio']), pd.Timestamp(ev['fim']), color='orange', alpha=0.25)
ax.set_ylabel('SSTA Nino 3.4 (C)')
ax.set_title('Eventos El Nino semanais (criterio ONI adaptado, Trenberth 1997)')
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout()
f3.save_table(pd.DataFrame({'ssta': ssta, 'ssta_suavizada': suave}), 'FigF3N2_serie_eventos_dados')
f3.save_figure(fig, 'FigF3N2_serie_eventos')
plt.close(fig)

In [ ]:
janela = 52
perfis = {}
for _, ev in catalogo.iterrows():
    pico = pd.Timestamp(ev['semana_pico'])
    trecho = suave.loc[pico - pd.Timedelta(weeks=janela):pico + pd.Timedelta(weeks=janela)]
    rel = ((trecho.index - pico).days / 7).astype(int)
    perfis[ev['evento']] = pd.Series(trecho.to_numpy(), index=rel)
matriz_perfis = pd.DataFrame(perfis).reindex(range(-janela, janela + 1))
matriz_perfis.index.name = 'semanas_relativas_ao_pico'
f3.save_table(matriz_perfis, 'FigF3N2_perfis_eventos_dados')

fig, ax = plt.subplots(figsize=(10, 5))
for evento in matriz_perfis.columns:
    ax.plot(matriz_perfis.index, matriz_perfis[evento], lw=0.8, alpha=0.6, label=evento)
ax.plot(matriz_perfis.index, matriz_perfis.mean(axis=1), color='k', lw=2.2, label='composto medio')
ax.axvline(0, color='k', lw=0.6, ls=':')
ax.axhline(0.5, color='r', lw=0.6, ls='--')
ax.set_xlabel('Semanas relativas ao pico')
ax.set_ylabel('SSTA suavizada (C)')
ax.set_title('Evolucao temporal dos eventos alinhados pelo pico')
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
f3.save_figure(fig, 'FigF3N2_perfis_eventos')
plt.close(fig)